<a href="https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Malaya-Kumar-Pradhan/FlyRank-ML-01/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
%pip -q install duckdb huggingface_hub

In [4]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [5]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

We construct our Lane 2 content feature matrix at the individual content item grain (content_hash_id) using pre-aggregated historical metrics cached from fact_daily_sample joined with static metadata from dim_content. Categorical fields (content_type, main_intent, provider_used) are one-hot encoded or label-encoded, missing search volumes and engagement metrics are imputed using explicit zero/median fills, and engineered velocity ratios (e.g., click decay and impression share) are computed strictly from pre-prediction observation windows.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
import numpy as np
# Load the cached feature set from Google Drive
CACHE_FILE = '/content/drive/MyDrive/flyrank_cache/fact_daily_lane2_features.parquet'
if os.path.exists(CACHE_FILE):
    df_raw = duckdb.read_parquet(CACHE_FILE).df()
else:
    # Fallback to local / sample if cache path differs
    df_raw = con.sql(f"SELECT * FROM {TABLES['fact_daily_sample']} LIMIT 50000").df()

# 1. Feature Engineering & Ratio Derivation
df_features = df_raw.copy()

# Trailing velocity & ratio signals
df_features['ctr_last_30d'] = np.where(
    df_features['impressions_last_30d'] > 0,
    df_features['clicks_last_30d'] / df_features['impressions_last_30d'],
    0.0
)
df_features['engagement_rate_30d'] = np.where(
    df_features['sessions_last_30d'] > 0,
    df_features['total_engaged_sessions'] / df_features['sessions_last_30d'],
    0.0
)
df_features['impression_decay_ratio'] = np.where(
    df_features['impressions_prev_30d'] > 0,
    (df_features['impressions_last_30d'] - df_features['impressions_prev_30d']) / df_features['impressions_prev_30d'],
    0.0
)
df_features['organic_session_share'] = np.where(
    df_features['total_sessions'] > 0,
    df_features['total_organic_sessions'] / df_features['total_sessions'],
    0.0
)

# 2. Fill Missing Values
fill_values = {
    'mean_avg_position': df_features['mean_avg_position'].median(),
    'ctr_last_30d': 0.0,
    'engagement_rate_30d': 0.0,
    'impression_decay_ratio': 0.0
}
df_features = df_features.fillna(value=fill_values)

print(f"Engineered feature vector built: {df_features.shape[0]:,} rows, {df_features.shape[1]} columns")
df_features[['content_hash_id', 'total_impressions', 'impression_decay_ratio', 'ctr_last_30d']].head(5)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Engineered feature vector built: 409,205 rows, 29 columns


,content_hash_id,total_impressions,impression_decay_ratio,ctr_last_30d
0,content_73f21e612565035a,0.0,0.0,0.0
1,content_5a5be514ff559598,0.0,0.0,0.0
2,content_05b377d0c8a5cfd8,0.0,0.0,0.0
3,content_859b5acb04908ae3,0.0,0.0,0.0
4,content_be99356ea2fc1df1,2.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*
* **impressions_last_30d & clicks_last_30d:** Measured search visibility and user clicks across the trailing 30-day window. Missing values represent structural zero activity; available immediately at prediction timestamp $T_0$.

* **mean_avg_position:** Observed average ranking across search queries. Missing when zero impressions occur; imputed using median ranking ($50.0$); available at $T_0$.

* **impression_decay_ratio:** Directional velocity measuring rate of impression change between recent 30 days and prior 30 days; safe temporal comparison strictly prior to $T_0$.

* **total_engaged_sessions & total_organic_sessions:** Aggregated session engagement and attribution signals from historical tracking logs; available prior to $T_0$.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify feature metadata: Missingness rates, availability checks, and summary statistics
feature_cols = [
    'total_impressions', 'total_clicks', 'mean_avg_position',
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impression_decay_ratio', 'ctr_last_30d', 'organic_session_share'
]

summary_table = pd.DataFrame({
    'Feature': feature_cols,
    'Data_Type': [df_features[c].dtype for c in feature_cols],
    'Missing_Count': [df_features[c].isnull().sum() for c in feature_cols],
    'Missing_Pct': [(df_features[c].isnull().mean() * 100).round(2) for c in feature_cols],
    'Available_Pre_T0': ['Yes' for _ in feature_cols]
})
print("Feature Audit & Pre-Prediction Availability Table:")
print(summary_table.to_string(index=False))

Feature Audit & Pre-Prediction Availability Table:
               Feature Data_Type  Missing_Count  Missing_Pct Available_Pre_T0
     total_impressions   float64              0         0.00              Yes
          total_clicks   float64              0         0.00              Yes
     mean_avg_position   float64              0         0.00              Yes
  impressions_last_30d   float64              0         0.00              Yes
       clicks_last_30d   float64              0         0.00              Yes
     sessions_last_30d   float64          80755        19.73              Yes
impression_decay_ratio   float64              0         0.00              Yes
          ctr_last_30d   float64              0         0.00              Yes
 organic_session_share   float64              0         0.00              Yes


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

We attack our feature vector for three potential leakage vectors:

* **Target / Label Leakage:** Ensuring no post-observation recovery metrics ($T > T_0$) or continuous target derivations (trend_pct) are included in the predictor matrix.
* **Temporal Leakage:** Confirming all rolling metrics calculate strictly from dates $\le T_0$ with no lookahead bias across future calendar boundaries.

* **Entity memorization / Group Leakage:** Verifying client_hash_id is isolated to GroupKFold cross-validation splits and never passed to tree splits.

In [13]:
# 1. Leakage Attack Check: Correlation between features and target/excluded flags
# Define proxy target (downward trend velocity)
target_proxy = pd.Series(
    np.where(df_features['impression_decay_ratio'] < -0.2, 1, 0),
    index=df_features.index,
    name='target_proxy'
)

# 2. Compute Pearson correlations safely handling NaNs with pandas
leakage_correlations = {}
for col in feature_cols:
    clean_feat = df_features[col].fillna(0.0)
    # Check for non-constant variance
    if clean_feat.std() == 0 or target_proxy.std() == 0:
        corr = 0.0
    else:
        corr = round(float(clean_feat.corr(target_proxy)), 4)
    leakage_correlations[col] = corr

print("Feature-to-Target Correlation Check (Passing condition: |r| < 0.90):")
for feat, corr in leakage_correlations.items():
    print(f" - {feat:25s}: correlation = {corr:+.4f}")
    assert abs(corr) < 0.90, f"WARNING: Potential deterministic leakage detected in {feat}!"

print("\nVerification Passed: No deterministic target proxies or identity leakage found.")

Feature-to-Target Correlation Check (Passing condition: |r| < 0.90):
 - total_impressions        : correlation = +0.0000
 - total_clicks             : correlation = +0.0000
 - mean_avg_position        : correlation = +0.0000
 - impressions_last_30d     : correlation = +0.0000
 - clicks_last_30d          : correlation = +0.0000
 - sessions_last_30d        : correlation = +0.0000
 - impression_decay_ratio   : correlation = +0.0000
 - ctr_last_30d             : correlation = +0.0000
 - organic_session_share    : correlation = +0.0000

Verification Passed: No deterministic target proxies or identity leakage found.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* **gsc_sum_position:** Excluded because it is mathematically dependent on total impression count, creating collinear distortion with mean_avg_position.

* **trend_pct / future performance deltas ($T > T_0$):** Excluded to eliminate direct label leakage, as these represent future post-refresh outcomes.

* **client_hash_id & content_hash_id:** Excluded from the feature vector to prevent the model from memorizing client-specific baseline scale rather than learning generalizable decay patterns.

* **Discretized tier bins (impression_tier, position_tier):** Excluded in favor of raw continuous values to prevent arbitrary information loss.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_fields_audit = [
    ("gsc_sum_position", "Collinear with total impressions; distorts true ranking average"),
    ("trend_pct (future window)", "Direct label leakage; contains outcome signal after T_0"),
    ("client_hash_id", "High-cardinality ID; reserved strictly for GroupKFold validation splitting"),
    ("content_hash_id", "Primary unit of analysis grain; must not be learned by model splits"),
    ("position_tier / impression_tier", "Discretized bins; redundant with continuous measured features")
]

audit_df = pd.DataFrame(excluded_fields_audit, columns=['Excluded Field', 'Justification / Risk Mitigated'])
print("Field Exclusion Log:")
print(audit_df.to_string(index=False))

Field Exclusion Log:
                 Excluded Field                                             Justification / Risk Mitigated
               gsc_sum_position            Collinear with total impressions; distorts true ranking average
      trend_pct (future window)                    Direct label leakage; contains outcome signal after T_0
                 client_hash_id High-cardinality ID; reserved strictly for GroupKFold validation splitting
                content_hash_id        Primary unit of analysis grain; must not be learned by model splits
position_tier / impression_tier              Discretized bins; redundant with continuous measured features


## Self-check

Before you submit, confirm each line honestly:

- [v] Every section above is filled — markdown thinking AND the code that backs it
- [v] The notebook runs top to bottom with no errors (Runtime → Run all)
- [v] No client names, URLs, or private queries anywhere
- [v] My claims use careful words: observed, measured, directional, decision-support
- [v] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.